# Mini Training Framework

把前七课的知识组装成一个 100 行内可复用的训练框架：Dataset → DataLoader → Model → Loss → Optimizer → Trainer。换模型/损失/优化器只需改一行。


## 0. 环境配置与导入


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib

matplotlib.rcParams["font.sans-serif"] = ["PingFang SC", "Hiragino Sans GB", "Arial Unicode MS", "Microsoft YaHei", "sans-serif"]
matplotlib.rcParams["axes.unicode_minus"] = False

print("PyTorch version:", torch.__version__)
torch.manual_seed(42)


## 1. 数据层：Dataset 与 DataLoader


In [ ]:
class Dataset:
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.float32).unsqueeze(1)
    def __len__(self):
        return len(self.y)
    def __getitem__(self, i):
        return self.X[i], self.y[i]

class DataLoader:
    def __init__(self, ds, batch_size, shuffle=True, seed=0):
        self.ds, self.bs, self.shuffle = ds, batch_size, shuffle
        self.rng = np.random.default_rng(seed)
    def __iter__(self):
        idx = np.arange(len(self.ds))
        if self.shuffle:
            self.rng.shuffle(idx)
        for i in range(0, len(idx), self.bs):
            sel = idx[i:i+self.bs]
            yield self.ds.X[sel], self.ds.y[sel]

rng = np.random.default_rng(42)
n = 400
X0 = rng.standard_normal((n, 2)) + np.array([-2.0, 0.0])
X1 = rng.standard_normal((n, 2)) + np.array([2.0, 0.0])
X = np.vstack([X0, X1]); y = np.concatenate([np.zeros(n), np.ones(n)])
idx = rng.permutation(len(X)); tr, va = idx[:320], idx[320:]

train_ds = Dataset(X[tr], y[tr]); val_ds = Dataset(X[va], y[va])
train_dl = DataLoader(train_ds, 32, shuffle=True)
print("首个 batch 形状:", next(iter(train_dl))[0].shape, "→ (batch, 特征数)")


## 2. 训练器：Trainer


In [ ]:
class Trainer:
    def __init__(self, model, opt, train_dl, val_dl=None, epochs=50):
        self.model, self.opt = model, opt
        self.train_dl, self.val_dl = train_dl, val_dl
        self.epochs = epochs
        self.history = {'train': [], 'val': [], 'val_acc': []}

    def fit(self):
        for epoch in range(self.epochs):
            self.model.train()
            ep = []
            for xb, yb in self.train_dl:
                self.opt.zero_grad()
                loss = F.binary_cross_entropy(torch.sigmoid(self.model(xb)), yb)
                loss.backward(); self.opt.step()
                ep.append(loss.item())
            self.history['train'].append(np.mean(ep))
            self.evaluate()

    def evaluate(self):
        self.model.eval()
        with torch.no_grad():
            out = torch.sigmoid(self.model(self.val_ds.X))
            loss = F.binary_cross_entropy(out, self.val_ds.y).item()
            acc = ((out.ravel() > 0.5) == self.val_ds.y.ravel()).float().mean().item()
        self.history['val'].append(loss); self.history['val_acc'].append(acc)

    def __getattr__(self, name):
        # 便捷访问：trainer.val_ds
        if name == 'val_ds' and self.val_dl is not None:
            return self.val_dl.ds
        raise AttributeError(name)


## 3. 用同一 Trainer 训练不同模型


In [ ]:
def train_and_report(make_model, make_opt, name):
    torch.manual_seed(0)
    model = make_model()
    trainer = Trainer(model, make_opt(model.parameters()), train_dl,
                      DataLoader(val_ds, 32, shuffle=False), epochs=80)
    trainer.fit()
    acc = trainer.history['val_acc'][-1]
    print(f"{name:<14} val 准确率 = {acc:.3f}")
    return trainer

# 同一个 Trainer，换模型/优化器只需改构造参数
t_mlp  = train_and_report(lambda: nn.Sequential(nn.Linear(2, 16), nn.ReLU(), nn.Linear(16, 1)),
                          lambda p: torch.optim.Adam(p, lr=0.01), 'MLP + Adam')
t_mlp_sgd = train_and_report(lambda: nn.Sequential(nn.Linear(2, 16), nn.ReLU(), nn.Linear(16, 1)),
                          lambda p: torch.optim.SGD(p, lr=0.1, momentum=0.9), 'MLP + Momentum')
t_linear = train_and_report(lambda: nn.Linear(2, 1),
                          lambda p: torch.optim.Adam(p, lr=0.01), 'Linear + Adam')

plt.figure(figsize=(8, 4.5))
for t, name in [(t_mlp, 'MLP+Adam'), (t_mlp_sgd, 'MLP+Momentum'), (t_linear, 'Linear+Adam')]:
    plt.plot(t.history['val_acc'], label=name)
plt.xlabel('epoch'); plt.ylabel('val 准确率')
plt.title('同一 Trainer、不同模型/优化器的可复用对比')
plt.legend(); plt.grid(alpha=0.3)


## 4. 扩展点：换损失、加正则


In [ ]:
# 扩展 1：换损失函数（MSE 用于回归任务）
class RegTrainer(Trainer):
    def fit(self):
        for epoch in range(self.epochs):
            self.model.train()
            ep = []
            for xb, yb in self.train_dl:
                self.opt.zero_grad()
                loss = F.mse_loss(self.model(xb), yb)          # 只改这一行
                loss.backward(); self.opt.step()
                ep.append(loss.item())
            self.history['train'].append(np.mean(ep))
            self.evaluate()

# 扩展 2：L2 正则（weight_decay）
torch.manual_seed(0)
reg_model = nn.Sequential(nn.Linear(2, 32), nn.ReLU(), nn.Linear(32, 1))
t_reg = Trainer(reg_model,
                torch.optim.Adam(reg_model.parameters(), lr=0.01, weight_decay=1e-3),
                train_dl, DataLoader(val_ds, 32, shuffle=False), epochs=80)
t_reg.fit()
print("加 weight_decay 的 val 准确率:", round(t_reg.history['val_acc'][-1], 3))


## 5. 从数学到框架的完整链路


回顾整个课程链：线性代数（矩阵如何表示变换）→ 微积分（梯度从哪来）→ 概率（损失 = 负对数似然）→ 神经网络（把这三者装进一个可训练的系统）。本课的 Trainer 就是最小化的 `PyTorch Lightning` / 自定义训练循环。

**工程化方向**（超出本课范围）：`torch.utils.data.DataLoader`（多进程加载）、`torch.compile`（加速）、分布式训练、checkpoint 与日志。


## 课后练习


1. **加 Early Stopping**：给 Trainer 加 `patience` 参数，val 不降则停止并恢复最优权重。
2. **加 Dropout**：构造带 Dropout 的模型用 Trainer 训练，对比泛化。
3. **回归任务**：造一个 $y = \sin(x) + \text{噪声}$ 的回归数据集，用 RegTrainer 训练并画拟合曲线。
4. **换优化器**：给 Trainer 加 `scheduler` 支持（每个 epoch 后 step）。
5. **思考**：Trainer 的哪些部分应该抽象、哪些应该暴露？对比你见过的训练框架设计。
